# berts-gg-qa-training

Offline Kaggle training notebook for **Google QUEST Q&A Labeling** using a dual-input transformer regressor.

- Internet: OFF
- Models loaded from Kaggle datasets only
- Trains with GroupKFold, AMP, optional SWA
- Saves fold checkpoints + OOF + postprocess artifacts


In [ ]:
import os, gc, json, html, random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import GroupKFold

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup


In [ ]:
@dataclass
class CFG:
    seed: int = 42
    n_folds: int = 5
    epochs: int = 3
    train_bs: int = 4
    valid_bs: int = 8
    grad_accum: int = 2
    lr: float = 2e-5
    wd: float = 0.01
    max_len_q: int = 256   # title + body
    max_len_a: int = 256   # title + answer
    num_workers: int = 2
    use_amp: bool = True
    use_swa: bool = True
    swa_start_epoch: int = 2
    out_dir: str = '/kaggle/working/ggqa_bert_models'

    model_zoo = {
        'roberta_base': '/kaggle/input/roberta-base',
        'roberta_large': '/kaggle/input/robertalarge',
        'deberta_v3_base': '/kaggle/input/debertav3base'
    }

cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


In [ ]:
train = pd.read_csv('/kaggle/input/google-quest-challenge/train.csv')
test = pd.read_csv('/kaggle/input/google-quest-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/google-quest-challenge/sample_submission.csv')

target_cols = [c for c in train.columns if c not in test.columns and c != 'qa_id']
question_targets = [c for c in target_cols if c.startswith('question_')]
answer_targets = [c for c in target_cols if c.startswith('answer_')]

train[['question_title','question_body','answer']] = train[['question_title','question_body','answer']].fillna('').apply(lambda c: c.map(html.unescape))
test[['question_title','question_body','answer']] = test[['question_title','question_body','answer']].fillna('').apply(lambda c: c.map(html.unescape))

y = train[target_cols].values.astype(np.float32)
label_values = {c: np.sort(train[c].unique()) for c in target_cols}
groups = train['question_body'].values

print(len(target_cols), len(question_targets), len(answer_targets))


In [ ]:
def mean_spearman(y_true, y_pred):
    scores = []
    for i in range(y_true.shape[1]):
        s = spearmanr(y_true[:, i], y_pred[:, i]).correlation
        scores.append(0.0 if np.isnan(s) else s)
    return float(np.mean(scores))

def snap_to_known_labels(preds, target_cols, label_values):
    out = np.zeros_like(preds)
    for j, c in enumerate(target_cols):
        vals = label_values[c]
        idx = np.abs(preds[:, [j]] - vals.reshape(1, -1)).argmin(axis=1)
        out[:, j] = vals[idx]
    return out

# Optional clipping map from public writeups/discussions
CLIPPINGS = {
    'question_has_commonly_accepted_answer': (0.0, 0.6),
    'question_conversational': (0.15, 1.0),
    'question_multi_intent': (0.1, 1.0),
    'question_type_choice': (0.1, 1.0),
    'question_type_compare': (0.1, 1.0),
    'question_type_consequence': (0.08, 1.0),
    'question_type_definition': (0.1, 1.0),
    'question_type_entity': (0.13, 1.0)
}

def apply_clipping(preds, cols, clippings=CLIPPINGS):
    out = preds.copy()
    for j, c in enumerate(cols):
        if c in clippings:
            lo, hi = clippings[c]
            out[:, j] = np.clip(out[:, j], lo, hi)
    return out


In [ ]:
class QADataset(Dataset):
    def __init__(self, df, y=None, tokenizer=None, max_len_q=256, max_len_a=256):
        self.title = df['question_title'].tolist()
        self.body = df['question_body'].tolist()
        self.answer = df['answer'].tolist()
        self.y = y
        self.tok = tokenizer
        self.max_len_q = max_len_q
        self.max_len_a = max_len_a

    def __len__(self):
        return len(self.title)

    def __getitem__(self, idx):
        q = self.tok(
            self.title[idx], self.body[idx],
            truncation=True, max_length=self.max_len_q,
            padding='max_length', return_tensors='pt'
        )
        a = self.tok(
            self.title[idx], self.answer[idx],
            truncation=True, max_length=self.max_len_a,
            padding='max_length', return_tensors='pt'
        )
        item = {
            'q_input_ids': q['input_ids'][0],
            'q_attention_mask': q['attention_mask'][0],
            'a_input_ids': a['input_ids'][0],
            'a_attention_mask': a['attention_mask'][0],
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y[idx], dtype=torch.float32)
        return item


In [ ]:
class DualTowerRegressor(nn.Module):
    def __init__(self, model_path, n_targets=30, q_idx=None, a_idx=None):
        super().__init__()
        self.q_encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        self.a_encoder = AutoModel.from_pretrained(model_path, local_files_only=True)

        hidden = self.q_encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, n_targets)
        )

        self.q_idx = q_idx
        self.a_idx = a_idx

    def pool(self, out, mask):
        cls = out[:, 0]
        mask = mask.unsqueeze(-1)
        mean = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return torch.cat([cls, mean], dim=1)

    def forward(self, q_input_ids, q_attention_mask, a_input_ids, a_attention_mask):
        q = self.q_encoder(input_ids=q_input_ids, attention_mask=q_attention_mask).last_hidden_state
        a = self.a_encoder(input_ids=a_input_ids, attention_mask=a_attention_mask).last_hidden_state

        q_vec = self.pool(q, q_attention_mask)
        a_vec = self.pool(a, a_attention_mask)

        # pool() returns 2*hidden each; use half from each to keep 2*hidden total
        h = torch.cat([q_vec[:, :q_vec.shape[1]//2], a_vec[:, :a_vec.shape[1]//2]], dim=1)
        h = self.dropout(h)
        out = self.head(h)
        return torch.sigmoid(out)


In [ ]:
def run_fold(model_name, model_path, fold, tr_idx, va_idx):
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

    tr_ds = QADataset(train.iloc[tr_idx].reset_index(drop=True), y[tr_idx], tokenizer, cfg.max_len_q, cfg.max_len_a)
    va_ds = QADataset(train.iloc[va_idx].reset_index(drop=True), y[va_idx], tokenizer, cfg.max_len_q, cfg.max_len_a)
    te_ds = QADataset(test.reset_index(drop=True), None, tokenizer, cfg.max_len_q, cfg.max_len_a)

    tr_loader = DataLoader(tr_ds, batch_size=cfg.train_bs, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    te_loader = DataLoader(te_ds, batch_size=cfg.valid_bs, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

    model = DualTowerRegressor(model_path, n_targets=len(target_cols)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    total_steps = len(tr_loader) * cfg.epochs // cfg.grad_accum
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    scaler = GradScaler(enabled=cfg.use_amp)
    crit = nn.SmoothL1Loss()

    swa_state = None
    best_score = -1
    best_path = f"{cfg.out_dir}/{model_name}_fold{fold}.pt"

    for ep in range(cfg.epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        for step, b in enumerate(tr_loader):
            q_ids = b['q_input_ids'].to(device)
            q_msk = b['q_attention_mask'].to(device)
            a_ids = b['a_input_ids'].to(device)
            a_msk = b['a_attention_mask'].to(device)
            lbl = b['labels'].to(device)

            with autocast(enabled=cfg.use_amp):
                pred = model(q_ids, q_msk, a_ids, a_msk)
                loss = crit(pred, lbl) / cfg.grad_accum

            scaler.scale(loss).backward()

            if (step + 1) % cfg.grad_accum == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        if cfg.use_swa and ep >= cfg.swa_start_epoch:
            cur = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if swa_state is None:
                swa_state = cur
            else:
                for k in swa_state:
                    swa_state[k] = 0.5 * swa_state[k] + 0.5 * cur[k]

        model.eval()
        va_pred = []
        with torch.no_grad():
            for b in va_loader:
                q_ids = b['q_input_ids'].to(device)
                q_msk = b['q_attention_mask'].to(device)
                a_ids = b['a_input_ids'].to(device)
                a_msk = b['a_attention_mask'].to(device)
                p = model(q_ids, q_msk, a_ids, a_msk).detach().cpu().numpy()
                va_pred.append(p)
        va_pred = np.concatenate(va_pred)
        score = mean_spearman(y[va_idx], va_pred)
        print(model_name, 'fold', fold, 'epoch', ep, 'cv', score)

        if score > best_score:
            best_score = score
            torch.save(model.state_dict(), best_path)

    if cfg.use_swa and swa_state is not None:
        torch.save(swa_state, best_path.replace('.pt', '_swa.pt'))

    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    oof = []
    with torch.no_grad():
        for b in va_loader:
            q_ids = b['q_input_ids'].to(device)
            q_msk = b['q_attention_mask'].to(device)
            a_ids = b['a_input_ids'].to(device)
            a_msk = b['a_attention_mask'].to(device)
            oof.append(model(q_ids, q_msk, a_ids, a_msk).detach().cpu().numpy())
    oof = np.concatenate(oof)

    test_pred = []
    with torch.no_grad():
        for b in te_loader:
            q_ids = b['q_input_ids'].to(device)
            q_msk = b['q_attention_mask'].to(device)
            a_ids = b['a_input_ids'].to(device)
            a_msk = b['a_attention_mask'].to(device)
            test_pred.append(model(q_ids, q_msk, a_ids, a_msk).detach().cpu().numpy())
    test_pred = np.concatenate(test_pred)

    del model, tr_loader, va_loader, te_loader, tr_ds, va_ds, te_ds
    gc.collect(); torch.cuda.empty_cache()

    return oof, test_pred, best_score, best_path


In [ ]:
gkf = GroupKFold(n_splits=cfg.n_folds)

all_model_oof = {}
all_model_test = {}

for model_name, model_path in cfg.model_zoo.items():
    print('\n===', model_name, '===')
    oof = np.zeros((len(train), len(target_cols)), dtype=np.float32)
    test_pred = np.zeros((len(test), len(target_cols)), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(train, y, groups=groups)):
        fold_oof, fold_test, fold_score, ckpt = run_fold(model_name, model_path, fold, tr_idx, va_idx)
        oof[va_idx] = fold_oof
        test_pred += fold_test / cfg.n_folds
        print('saved', ckpt)

    all_model_oof[model_name] = oof
    all_model_test[model_name] = test_pred

ensemble_oof = np.mean(np.stack(list(all_model_oof.values()), axis=0), axis=0)
ensemble_test = np.mean(np.stack(list(all_model_test.values()), axis=0), axis=0)

print('Ensemble raw CV:', mean_spearman(y, ensemble_oof))


In [ ]:
# Postprocess variants for validation
pp_clip = apply_clipping(ensemble_oof, target_cols)
pp_clip_snap = snap_to_known_labels(pp_clip, target_cols, label_values)

print('CV clip:', mean_spearman(y, pp_clip))
print('CV clip+snap:', mean_spearman(y, pp_clip_snap))

# choose robust final: clip + snap
final_test = snap_to_known_labels(apply_clipping(ensemble_test, target_cols), target_cols, label_values)

# verify support
invalid = {}
for c in target_cols:
    invalid[c] = int((~pd.Series(final_test[:, target_cols.index(c)]).isin(label_values[c])).sum())
print('max invalid', max(invalid.values()), 'total invalid', sum(invalid.values()))


In [ ]:
os.makedirs(f"{cfg.out_dir}/artifacts", exist_ok=True)

np.save(f"{cfg.out_dir}/artifacts/oof_ensemble.npy", ensemble_oof)
np.save(f"{cfg.out_dir}/artifacts/test_ensemble.npy", ensemble_test)
np.save(f"{cfg.out_dir}/artifacts/test_final_pp.npy", final_test)

with open(f"{cfg.out_dir}/artifacts/target_cols.json", 'w') as f:
    json.dump(target_cols, f)
with open(f"{cfg.out_dir}/artifacts/label_values.json", 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in label_values.items()}, f)
with open(f"{cfg.out_dir}/artifacts/clippings.json", 'w') as f:
    json.dump({k: [float(v[0]), float(v[1])] for k, v in CLIPPINGS.items()}, f)

# convenience copies at root (so submission notebook can work even if only root files are uploaded)
with open(f"{cfg.out_dir}/target_cols.json", 'w') as f:
    json.dump(target_cols, f)
with open(f"{cfg.out_dir}/label_values.json", 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in label_values.items()}, f)
with open(f"{cfg.out_dir}/clippings.json", 'w') as f:
    json.dump({k: [float(v[0]), float(v[1])] for k, v in CLIPPINGS.items()}, f)

sub = sample.copy()
sub[target_cols] = final_test
sub.to_csv(f"{cfg.out_dir}/artifacts/submission_train_notebook.csv", index=False)
print('saved artifacts to', f"{cfg.out_dir}/artifacts")
sub.head()
